# Finite filtered-jump weak channel

This companion demonstrates the original restricted zero-C channel. It retains both weak outcomes and every environment label. Finite mesh bias, long-circuit channel powers and physical estimates have separate scopes. The general sampler remains planned.

In [1]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/openquantum').is_dir())
sys.path.insert(0, str(root / 'src'))
from openquantum.simulation import detailed_balance_channel as dbc
terms, proposals = ((1.0, 'Z'),), ('X', 'Z')

In [2]:
# A three-wire actual primitive demo, with one proposal and one clock bit.
small = dbc.build_weak_dilation(terms, ('X',), clock_bits=1, delta=0.125)
assert small.num_qubits == 3
print(small.draw('text', fold=80))

     ┌──────────────┐┌───┐┌─────────────┐┌───┐┌──────────────┐┌───┐»
q_0: ┤ Rz(-0.44311) ├┤ X ├┤ Rz(0.44311) ├┤ X ├┤ Rz(-0.44311) ├┤ X ├»
     ├─────────────┬┘└─┬─┘└─────────────┘└─┬─┘└──────────────┘└─┬─┘»
q_1: ┤ Ry(0.85556) ├───■───────────────────■────────────────────■──»
     ├─────────────┤                                               »
q_2: ┤ Ry(0.59016) ├───────────────────────────────────────────────»
     └─────────────┘                                               »
«     ┌─────────────┐┌───┐┌───┐┌───┐┌──────────────┐┌───┐┌─────────────┐┌───┐»
«q_0: ┤ Rz(0.44311) ├┤ X ├┤ X ├┤ X ├┤ Rz(-0.44311) ├┤ X ├┤ Rz(0.44311) ├┤ X ├»
«     └─────────────┘└─┬─┘└───┘└─┬─┘└──────────────┘└─┬─┘└─────────────┘└─┬─┘»
«q_1: ─────────────────■─────────■────────────────────■───────────────────■──»
«                                                                            »
«q_2: ───────────────────────────────────────────────────────────────────────»
«                                          

In [3]:
checked = dbc.channel_check(terms, proposals, clock_bits=2, delta=0.125, rounds=64)
assert checked['actual_vs_independent_channel_frobenius_error'] < 1e-10
assert checked['trace_preservation_max_entry_error'] < 1e-10
assert checked['minimum_choi_eigenvalue'] >= -1e-10
{k: checked[k] for k in ('actual_vs_independent_kraus_max_entry_error', 'nonzero_environment_nojump_weight', 'finite_round_gibbs_residual', 'finite_stationary_gibbs_trace_distance', 'bloch_contraction_norm')}

{'actual_vs_independent_kraus_max_entry_error': 1.432189986210916e-14,
 'nonzero_environment_nojump_weight': 0.000608039969191247,
 'finite_round_gibbs_residual': 0.004901392359387127,
 'finite_stationary_gibbs_trace_distance': 0.051333124975567024,
 'bloch_contraction_norm': 0.9324839900898618}

In [4]:
# These are powers of the fully checked reduced channel, not simulation of
# the entire 64-round density circuit. All four spanning inputs are retained.
checked['spanning_input_channel_power_checks']

[{'trace_error': 1.9168000520180752e-12,
  'minimum_eigenvalue': 0.17999506603395188,
  'gibbs_trace_distance': 0.06079214401279266,
  'finite_stationary_trace_distance': 0.009459019037225638,
  'numerical_contraction_estimate_to_finite_stationary': 0.009459019036812837},
 {'trace_error': 1.916466985108016e-12,
  'minimum_eigenvalue': 0.16859129254583782,
  'gibbs_trace_distance': 0.04938837052467844,
  'finite_stationary_trace_distance': 0.0019447544508885836,
  'numerical_contraction_estimate_to_finite_stationary': 0.0019447544517386373},
 {'trace_error': 1.9192425426704302e-12,
  'minimum_eigenvalue': 0.17429270745930558,
  'gibbs_trace_distance': 0.05509304677579695,
  'finite_stationary_trace_distance': 0.0037978151841503876,
  'numerical_contraction_estimate_to_finite_stationary': 0.006828437266949522},
 {'trace_error': 1.9154677843867063e-12,
  'minimum_eigenvalue': 0.17429309800743178,
  'gibbs_trace_distance': 0.05509073782812675,
  'finite_stationary_trace_distance': 0.003764

In [5]:
# The source includes system initialization, both environmental reset
# boundaries, every actual round and final readout. Rotations are unsynthesized.
preparation = dbc.build_preparation(terms, proposals, clock_bits=2, delta=0.125, rounds=8)
assert preparation.count_ops()['reset'] == preparation.metadata['total_resets']
assert preparation.count_ops()['measure'] == 1
{'qubits': preparation.num_qubits, 'primitive_counts': dict(preparation.count_ops()), 'rounds': 8, 'physical_resources_computed': False, 'general_sampler_implemented': False}

{'qubits': 5,
 'primitive_counts': {'cx': 1920,
  't': 800,
  'tdg': 600,
  'h': 520,
  'rz': 384,
  'ry': 184,
  'p': 96,
  'reset': 65,
  'x': 32,
  'measure': 1},
 'rounds': 8,
 'physical_resources_computed': False,
 'general_sampler_implemented': False}

The independent continuum Lindbladian has the target Gibbs state stationary; the finite filter need not. The declared diagonal Hamiltonian and Pauli proposals make the coherent correction exactly zero. A noncommuting system needs an implemented correction circuit. No synthesis, physical model, hardware run or mechanized proof is supplied. Source limits permit larger circuits, while numerical checks stop at one system wire and seven total wires.

Reference: Chen et al., *Efficient quantum thermal simulation*, DOI 10.1038/s41586-025-09583-x. Original source uses the project Qiskit/NumPy/SciPy lock; no upstream benchmark has been reproduced.